In [7]:
import pandas as pd

# 1. Asignas el identificador que acabamos de ubicar en la imagen
dataset_id = "jbjy-vk9h"
sample_rows = 200_000

# 2. Python construye la source_url automáticamente usando tu dataset_id
source_url = f"https://www.datos.gov.co/resource/{dataset_id}.csv?$limit={sample_rows}"

# 3. Descarga la muestra y la guarda en la estructura de carpetas local
df_secop = pd.read_csv(source_url, low_memory=False)
df_secop.to_csv("data/raw/secop_sample.csv", index=False)

print(f"Filas descargadas: {len(df_secop):,}")
print(f"Columnas: {df_secop.shape[1]}")

Filas descargadas: 200,000
Columnas: 85


In [1]:
import os
import pandas as pd

# 1. Aseguramos que la carpeta exista
os.makedirs("data/raw", exist_ok=True)

# 2. Asignamos el identificador del IDEAM
dataset_id_ideam = "57sv-p2fu"
sample_rows = 150_000  # Límite ajustado para mayor velocidad

# 3. Construimos la URL
source_url_ideam = f"https://www.datos.gov.co/resource/{dataset_id_ideam}.csv?$limit={sample_rows}"

# 4. Descargamos y guardamos con mensajes de progreso
print("1. Conectando con datos.gov.co y descargando datos del IDEAM...")
df_ideam = pd.read_csv(source_url_ideam, low_memory=False)

print("2. Guardando archivo localmente en data/raw/ideam_sample.csv...")
df_ideam.to_csv("data/raw/ideam_sample.csv", index=False)

print("\n--- ¡Descarga e inspección finalizadas! ---")
print(f"Filas descargadas: {len(df_ideam):,}")
print(f"Columnas: {df_ideam.shape[1]}")

1. Conectando con datos.gov.co y descargando datos del IDEAM...
2. Guardando archivo localmente en data/raw/ideam_sample.csv...

--- ¡Descarga e inspección finalizadas! ---
Filas descargadas: 150,000
Columnas: 13


In [3]:
import os

def get_file_size_gb(file_path):
    """Tamaño real del archivo en disco, en gigabytes."""
    return os.path.getsize(file_path) / (1024 ** 3)

In [5]:
import pandas as pd

def measure_expansion_factor(file_path, **read_options):
    """Mide cuántas veces crece un archivo al cargarse en memoria."""
    size_disk_bytes = os.path.getsize(file_path)
    df = pd.read_csv(file_path, **read_options)
    size_memory_bytes = df.memory_usage(deep=True).sum()
    return size_memory_bytes / size_disk_bytes, df

In [13]:
import psutil

def get_available_memory_gb():
    """Memoria realmente disponible, descontando la que ya está en uso."""
    return psutil.virtual_memory().available / (1024 ** 3)

In [14]:
def profile_source(df, source_name):
    """Perfil mínimo de una fuente cargada."""
    dtype_counts = df.dtypes.value_counts().to_dict()
    object_columns = (df.dtypes == "object").sum()

    return {
        "fuente": source_name,
        "filas": len(df),
        "columnas": df.shape[1],
        "columnas_texto": int(object_columns),
        "proporcion_texto": round(object_columns / df.shape[1], 3),
        "tipos": {str(key): int(value) for key, value in dtype_counts.items()},
    }

In [17]:
import os
import pandas as pd

# Aseguramos que la carpeta resultados exista antes de guardar
os.makedirs("resultados", exist_ok=True)

sources = {
    "SECOP II": "data/raw/secop_sample.csv",
    "IDEAM": "data/raw/ideam_sample.csv",
    "GEIH": "data/raw/geih/Ocupados.csv", # Asegúrate de que este nombre sea el correcto
}

measurements = []
for source_name, file_path in sources.items():
    # --- AQUÍ ESTÁ EL CAMBIO: Agregamos encoding="latin-1" ---
    k_value, df = measure_expansion_factor(file_path, low_memory=False, encoding="latin-1")
    
    profile = profile_source(df, source_name)
    profile["tamano_disco_gb"] = round(get_file_size_gb(file_path), 4)
    profile["k"] = round(k_value, 2)
    measurements.append(profile)
    del df  # libere memoria antes de cargar la siguiente fuente

df_measurements = pd.DataFrame(measurements)
df_measurements.to_csv("resultados/mediciones.csv", index=False)
print(df_measurements[["fuente", "filas", "columnas", "proporcion_texto", "k"]])

     fuente   filas  columnas  proporcion_texto     k
0  SECOP II  200000        85             0.765  3.02
1     IDEAM  150000        13             0.615  2.87
2      GEIH   29611         1             1.000  1.13


In [18]:
def verify_level_1(df_measurements):
    """Verificación automática del nivel 1. Devuelve True si todo pasa."""
    checks = {
        "Hay exactamente 3 fuentes medidas": len(df_measurements) == 3,
        "Todos los k son mayores que 1": (df_measurements["k"] > 1).all(),
        "Los tres k son distintos": df_measurements["k"].nunique() == 3,
        "No hay tamaños de disco en cero": (df_measurements["tamano_disco_gb"] > 0).all(),
    }
    for description, passed in checks.items():
        print(f"{'PASA ' if passed else 'FALLA'} · {description}")
    return all(checks.values())

verify_level_1(df_measurements)

PASA  · Hay exactamente 3 fuentes medidas
PASA  · Todos los k son mayores que 1
PASA  · Los tres k son distintos
PASA  · No hay tamaños de disco en cero


True

In [19]:
import math

def compute_threshold_periods(memory_useful_gb, expansion_factor,
                              initial_size_gb, growth_rate):
    """Períodos que faltan para que la fuente exceda la memoria útil.

    Un resultado negativo indica que el umbral ya fue superado.
    """
    if growth_rate <= 0:
        raise ValueError("La tasa de crecimiento debe ser mayor que cero.")

    ratio = memory_useful_gb / (expansion_factor * initial_size_gb)
    return math.log(ratio) / math.log(1 + growth_rate)

In [20]:
# Tasa de crecimiento hipotética para el ejercicio (5% por período)
tasa_crecimiento = 0.05

# Estos son los datos reales que sacaste en tu archivo resultados.csv
fuentes = {
    "SECOP II": {"k": 3.02, "S0": 0.3045},
    "IDEAM": {"k": 2.87, "S0": 0.0321},
    "GEIH": {"k": 1.13, "S0": 0.0097}
}

print("=== ESCENARIO 1: Equipo de 8GB (Asumiendo 4.5GB útiles) ===")
for nombre, datos in fuentes.items():
    periodos = compute_threshold_periods(
        memory_useful_gb=4.5,            # <--- AQUÍ PONES EL PRIMER VALOR DESCONTADO
        expansion_factor=datos["k"],
        initial_size_gb=datos["S0"],
        growth_rate=tasa_crecimiento
    )
    print(f"{nombre}: Faltan {periodos:.1f} períodos para saturar la RAM")


print("\n=== ESCENARIO 2: Equipo de 16GB (Asumiendo 11GB útiles) ===")
for nombre, datos in fuentes.items():
    periodos = compute_threshold_periods(
        memory_useful_gb=11.0,           # <--- AQUÍ PONES EL SEGUNDO VALOR DESCONTADO
        expansion_factor=datos["k"],
        initial_size_gb=datos["S0"],
        growth_rate=tasa_crecimiento
    )
    print(f"{nombre}: Faltan {periodos:.1f} períodos para saturar la RAM")

=== ESCENARIO 1: Equipo de 8GB (Asumiendo 4.5GB útiles) ===
SECOP II: Faltan 32.5 períodos para saturar la RAM
IDEAM: Faltan 79.7 períodos para saturar la RAM
GEIH: Faltan 123.3 períodos para saturar la RAM

=== ESCENARIO 2: Equipo de 16GB (Asumiendo 11GB útiles) ===
SECOP II: Faltan 50.9 períodos para saturar la RAM
IDEAM: Faltan 98.0 períodos para saturar la RAM
GEIH: Faltan 141.7 períodos para saturar la RAM


In [21]:
# Análisis de sensibilidad
growth_scenarios = [0.01, 0.02, 0.04, 0.08, 0.16]

for growth_rate in growth_scenarios:
    periods = compute_threshold_periods(
        memory_useful_gb=12, # Reemplaza por tu M medida
        expansion_factor=5.4, # Reemplaza por el k de una de tus fuentes (ej. 3.02)
        initial_size_gb=7.8, # Reemplaza por el tamaño real de tu fuente
        growth_rate=growth_rate,
    )
    print(f"g = {growth_rate:.0%}  →  t_umbral = {periods:6.1f} períodos")

g = 1%  →  t_umbral = -126.2 períodos
g = 2%  →  t_umbral =  -63.4 períodos
g = 4%  →  t_umbral =  -32.0 períodos
g = 8%  →  t_umbral =  -16.3 períodos
g = 16%  →  t_umbral =   -8.5 períodos
